# MMI Laser Scan Analysis

This notebook is the single analysis script for this folder.

## Project rules
1. Keep the full workflow in this notebook.
2. Read the original `.loglaserscan_ymlz` file directly.
3. Keep the steps in order: load data, find local peaks and deeps, fit IL, calculate ER, then plot.
4. Use simple variable names and short comments so a beginner can follow the code.
5. If you want to tune the analysis, only change the parameter cell.


## What this notebook does

- Load the laser scan data from the original measurement file.
- Find local peaks and local deeps for `gc1->gc2` and `gc1->gc3`.
- Fit smooth peak and deep envelopes.
- Convert those envelopes to insertion loss (IL).
- Calculate extinction ratio (ER) from the fitted IL curves.
- Draw four subplots for an easy final check.

In this notebook, **deep** means local minimum.


In [ ]:
from pathlib import Path
import io
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from scipy.signal import find_peaks

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
data_files = sorted(Path('.').glob('*.loglaserscan_ymlz'))

if not data_files:
    raise FileNotFoundError('No .loglaserscan_ymlz file was found in this folder.')

DATA_FILE = data_files[0]
CHANNELS = ['gc1->gc2', 'gc1->gc3']

print(f'Using data file: {DATA_FILE.name}')


def load_loglaserscan_ymlz(file_path):
    """Load the text header and the embedded NumPy arrays from a .loglaserscan_ymlz file."""
    raw_bytes = Path(file_path).read_bytes()
    zip_start = raw_bytes.find(b'PK\x03\x04')

    if zip_start == -1:
        raise ValueError('Could not find the embedded ZIP data in the file.')

    header_text = raw_bytes[:zip_start].decode('utf-8', errors='replace')

    wav_match = re.search(
        r'wav: # xdata\s+first: ([0-9.]+)\s+last: ([0-9.]+)\s+points: (\d+)',
        header_text,
    )

    if wav_match is None:
        raise ValueError('Could not read the wavelength axis from the file header.')

    start_nm = float(wav_match.group(1))
    stop_nm = float(wav_match.group(2))
    point_count = int(wav_match.group(3))
    wavelength_nm = np.linspace(start_nm, stop_nm, point_count)

    scan_data = {'wavelength_nm': wavelength_nm}

    with zipfile.ZipFile(io.BytesIO(raw_bytes[zip_start:])) as zip_file:
        for file_name in zip_file.namelist():
            if file_name.endswith('.npy'):
                channel_name = file_name[:-4]
                scan_data[channel_name] = np.load(io.BytesIO(zip_file.read(file_name)))

    return header_text, scan_data


header_text, scan_data = load_loglaserscan_ymlz(DATA_FILE)
wavelength_nm = scan_data['wavelength_nm']

file_summary = pd.DataFrame(
    {
        'Channel': CHANNELS,
        'Points': [len(scan_data[channel]) for channel in CHANNELS],
        'Min (dB)': [scan_data[channel].min() for channel in CHANNELS],
        'Max (dB)': [scan_data[channel].max() for channel in CHANNELS],
    }
)

file_summary


## Analysis settings

These are the only settings you should normally change.

- `prominence_db`: how strong a peak or deep must be before we keep it.
- `min_distance_points`: minimum gap between two markers.
- `marker_smoothing_window`: small moving-average window used before the smooth fit.


In [ ]:
prominence_db = 1.0
min_distance_points = 250
marker_smoothing_window = 7


def moving_average(values, window):
    """Smooth a short list of marker values with a simple moving average."""
    values = np.asarray(values, dtype=float)

    if window <= 1 or len(values) < 3:
        return values.copy()

    half_window = window // 2
    smoothed = np.empty_like(values)

    for index in range(len(values)):
        left = max(0, index - half_window)
        right = min(len(values), index + half_window + 1)
        smoothed[index] = values[left:right].mean()

    return smoothed


def smooth_envelope(marker_x, marker_y, x_full, window):
    """Create a smooth curve from the marker points and keep flat edges outside the marker range."""
    marker_y_smooth = moving_average(marker_y, window)
    interpolator = PchipInterpolator(marker_x, marker_y_smooth, extrapolate=False)
    y_full = np.asarray(interpolator(x_full), dtype=float)

    y_full[x_full < marker_x[0]] = marker_y_smooth[0]
    y_full[x_full > marker_x[-1]] = marker_y_smooth[-1]

    return y_full


def analyze_channel(wavelength_nm, trace_db, prominence_db, min_distance_points, marker_smoothing_window):
    """Find local peaks and deeps, then build IL and ER curves."""
    peak_index, _ = find_peaks(trace_db, prominence=prominence_db, distance=min_distance_points)
    deep_index, _ = find_peaks(-trace_db, prominence=prominence_db, distance=min_distance_points)

    if len(peak_index) < 2 or len(deep_index) < 2:
        raise ValueError('Not enough local peaks or deeps were found. Try smaller detection settings.')

    peak_fit_db = smooth_envelope(
        wavelength_nm[peak_index],
        trace_db[peak_index],
        wavelength_nm,
        marker_smoothing_window,
    )
    deep_fit_db = smooth_envelope(
        wavelength_nm[deep_index],
        trace_db[deep_index],
        wavelength_nm,
        marker_smoothing_window,
    )

    # IL is reported as a positive loss number.
    peak_il_db = -peak_fit_db
    deep_il_db = -deep_fit_db
    er_db = deep_il_db - peak_il_db

    return {
        'peak_index': peak_index,
        'deep_index': deep_index,
        'peak_fit_db': peak_fit_db,
        'deep_fit_db': deep_fit_db,
        'peak_il_db': peak_il_db,
        'deep_il_db': deep_il_db,
        'er_db': er_db,
    }


results = {
    channel: analyze_channel(
        wavelength_nm=wavelength_nm,
        trace_db=scan_data[channel],
        prominence_db=prominence_db,
        min_distance_points=min_distance_points,
        marker_smoothing_window=marker_smoothing_window,
    )
    for channel in CHANNELS
}

analysis_summary = pd.DataFrame(
    {
        'Channel': CHANNELS,
        'Peak count': [len(results[channel]['peak_index']) for channel in CHANNELS],
        'Deep count': [len(results[channel]['deep_index']) for channel in CHANNELS],
        'Average peak IL (dB)': [results[channel]['peak_il_db'].mean() for channel in CHANNELS],
        'Average deep IL (dB)': [results[channel]['deep_il_db'].mean() for channel in CHANNELS],
        'Average ER (dB)': [results[channel]['er_db'].mean() for channel in CHANNELS],
    }
)

analysis_summary


In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
ax_raw, ax_gc2, ax_gc3, ax_er = axes.flat

# Subplot 1: raw data
for channel in CHANNELS:
    ax_raw.plot(wavelength_nm, scan_data[channel], linewidth=1.2, label=channel)

ax_raw.set_title('Raw data')
ax_raw.set_ylabel('Transmission (dB)')
ax_raw.legend()

# Subplot 2 and 3: markers and fitted envelopes on top of raw data
channel_axes = {'gc1->gc2': ax_gc2, 'gc1->gc3': ax_gc3}

for channel, axis in channel_axes.items():
    trace_db = scan_data[channel]
    channel_result = results[channel]

    axis.plot(wavelength_nm, trace_db, color='0.80', linewidth=1.0, label='Raw data')
    axis.scatter(
        wavelength_nm[channel_result['peak_index']],
        trace_db[channel_result['peak_index']],
        s=18,
        color='tab:red',
        label='Local peaks',
    )
    axis.scatter(
        wavelength_nm[channel_result['deep_index']],
        trace_db[channel_result['deep_index']],
        s=18,
        color='tab:blue',
        label='Local deeps',
    )
    axis.plot(wavelength_nm, channel_result['peak_fit_db'], color='tab:red', linewidth=2.0, label='Peak fit')
    axis.plot(wavelength_nm, channel_result['deep_fit_db'], color='tab:blue', linewidth=2.0, label='Deep fit')
    axis.set_title(channel)
    axis.set_ylabel('Transmission (dB)')
    axis.legend(fontsize=8)

# Subplot 4: ER
for channel in CHANNELS:
    ax_er.plot(wavelength_nm, results[channel]['er_db'], linewidth=2.0, label=channel)

ax_er.set_title('Extinction ratio (ER)')
ax_er.set_ylabel('ER (dB)')
ax_er.legend()

for axis in axes[1]:
    axis.set_xlabel('Wavelength (nm)')

figure.suptitle(f'MMI analysis for {DATA_FILE.name}', fontsize=14)
figure.tight_layout()

output_figure = DATA_FILE.with_name(f'{DATA_FILE.stem}_analysis_summary.png')
figure.savefig(output_figure, dpi=200, bbox_inches='tight')
print(f'Saved figure: {output_figure.name}')

plt.show()
